# Grover Search in 10 Minutes: Find a Row with Qiskit

**VLDB 2026 hands-on mini-lab** · No quantum background or IBM account required

> **Goal:** make a prediction, watch a quantum state change, discover the best iteration count, and sample the answer.

Run **Runtime → Run all**, then work through the challenges. Answers and measurements appear only when you click.

## 1. Setup

In [ ]:
%pip -q install "qiskit==2.5.0" "qiskit-aer==0.17.2" pylatexenc

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # Allows the same notebook to run in local Jupyter.

import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import math
import io
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

plt.style.use('seaborn-v0_8-whitegrid')
print('Ready: interactive widgets + Qiskit Aer simulator')

## 2. 🎯 Challenge 1 — Choose what Grover should find

Choose a database row. The notebook converts its row ID into the basis state that the oracle will mark.

Grover searches **row IDs**, not values directly. Three qubits label eight rows as $|000\rangle$ through $|111\rangle$.

In [ ]:
rows = [
    {'id': 0, 'city': 'Lima',  'sales': 41},
    {'id': 1, 'city': 'Oslo',  'sales': 67},
    {'id': 2, 'city': 'Tokyo', 'sales': 52},
    {'id': 3, 'city': 'Accra', 'sales': 76},
    {'id': 4, 'city': 'Paris', 'sales': 35},
    {'id': 5, 'city': 'Seoul', 'sales': 93},
    {'id': 6, 'city': 'Quito', 'sales': 58},
    {'id': 7, 'city': 'Delhi', 'sales': 49},
]
n, N = 3, 8
target_dropdown = widgets.Dropdown(
    options=[(f"{r['id']} — {r['city']} — sales={r['sales']}", r['id']) for r in rows],
    value=5, description='Target:', style={'description_width': 'initial'}
)
target_status = widgets.HTML()

def current_target():
    marked_id = target_dropdown.value
    return marked_id, format(marked_id, f'0{n}b')

def update_target_status(change=None):
    marked_id, marked_bits = current_target()
    target_status.value = (
        f"<b>Selected row:</b> <code>{rows[marked_id]}</code><br>"
        f"<b>Oracle target:</b> |{marked_bits}⟩"
    )

target_dropdown.observe(update_target_status, names='value')
display(widgets.VBox([target_dropdown, target_status]))
update_target_status()

## 3. 🔮 Predict what the oracle does

Before running diffusion, what happens to the target's measurement probability when the oracle flips its amplitude from $+1/\sqrt{8}$ to $-1/\sqrt{8}$?

In [ ]:
prediction = widgets.RadioButtons(
    options=['It increases', 'It decreases', 'It stays the same'], value=None
)
check_prediction = widgets.Button(description='Check prediction', button_style='primary')
prediction_feedback = widgets.HTML()

def check_oracle_prediction(_):
    if prediction.value is None:
        prediction_feedback.value = 'Choose an answer first.'
        return
    verdict = ('<h3>✅ Correct</h3>' if prediction.value == 'It stays the same' else
               '<h3>Not quite — look at what happens when an amplitude changes sign.</h3>')
    prediction_feedback.value = (
        verdict + '<p>Probability is |amplitude|², so +a and −a have the same probability. '
        'The oracle changes <b>phase</b>; diffusion turns that phase difference into amplitude amplification.</p>'
    )

check_prediction.on_click(check_oracle_prediction)
display(widgets.VBox([prediction, check_prediction, prediction_feedback]))

## 4. 👀 Watch one Grover iteration

Step manually through uniform superposition → oracle phase flip → diffusion (inversion about the mean).

In [ ]:
stage_selector = widgets.ToggleButtons(
    options=['Uniform', 'Oracle', 'Diffusion'], value='Uniform', description='Stage:'
)
play_stages = widgets.Button(description='▶ Play')
amplitude_image = widgets.Image(format='png', layout=widgets.Layout(width='100%'))
amplitude_stats = widgets.HTML()

def stage_data(marked_id):
    uniform = np.ones(N) / np.sqrt(N)
    oracle = uniform.copy()
    oracle[marked_id] *= -1
    diffusion = 2 * oracle.mean() - oracle
    return {'Uniform': uniform, 'Oracle': oracle, 'Diffusion': diffusion}

def draw_amplitude_stage(change=None):
    marked_id, marked_bits = current_target()
    stage = stage_selector.value
    amps = stage_data(marked_id)[stage]
    labels = [format(i, f'0{n}b') for i in range(N)]
    colors = ['#ef8354' if i == marked_id else '#4f6d9a' for i in range(N)]
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.bar(labels, amps, color=colors)
    ax.axhline(0, color='black', linewidth=.8)
    ax.set(ylim=(-0.5, 1.0), xlabel='row ID (binary)', ylabel='amplitude',
           title=f'{stage} stage — target |{marked_bits}⟩')
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', dpi=120)
    plt.close(fig)
    amplitude_image.value = buffer.getvalue()
    note = {'Uniform': 'All rows begin equally likely.',
            'Oracle': '<b>Same probability, opposite phase.</b>',
            'Diffusion': '<b>Inversion about the mean amplifies the marked state.</b>'}[stage]
    amplitude_stats.value = (
        f"<b>Target amplitude:</b> {amps[marked_id]:.4f}<br>"
        f"<b>Target probability:</b> {amps[marked_id]**2:.3f}<br>"
        f"<b>Mean amplitude:</b> {amps.mean():.4f}<br>{note}"
    )

def play_stage_sequence(_):
    for stage in stage_selector.options:
        stage_selector.value = stage
        time.sleep(0.8)

stage_selector.observe(draw_amplitude_stage, names='value')
target_dropdown.observe(draw_amplitude_stage, names='value')
play_stages.on_click(play_stage_sequence)
display(widgets.VBox([widgets.HBox([stage_selector, play_stages]), amplitude_image, amplitude_stats]))
draw_amplitude_stage()
display(widgets.HTML('<b>Target probability:</b> 0.125 → 0.125 → 0.781'))

### Implementation used by the interactive cells

These helpers construct the actual phase oracle, diffuser, and complete Grover circuit. You do not need to edit them.

In [ ]:
def phase_oracle(n, marked_bits):
    oracle = QuantumCircuit(n, name=f'Oracle {marked_bits}')
    zero_qubits = [q for q, bit in enumerate(reversed(marked_bits)) if bit == '0']
    if zero_qubits:
        oracle.x(zero_qubits)
    oracle.h(n - 1)
    oracle.mcx(list(range(n - 1)), n - 1)
    oracle.h(n - 1)
    if zero_qubits:
        oracle.x(zero_qubits)
    return oracle

def diffuser(n):
    diffusion = QuantumCircuit(n, name='Diffusion')
    diffusion.h(range(n)); diffusion.x(range(n)); diffusion.h(n - 1)
    diffusion.mcx(list(range(n - 1)), n - 1)
    diffusion.h(n - 1); diffusion.x(range(n)); diffusion.h(range(n))
    return diffusion

def grover_circuit(n, marked_bits, iterations, measure=False):
    circuit = QuantumCircuit(n)
    circuit.h(range(n)); circuit.barrier(label='uniform')
    for _ in range(iterations):
        circuit.compose(phase_oracle(n, marked_bits), inplace=True)
        circuit.compose(diffuser(n), inplace=True)
        circuit.barrier()
    if measure:
        circuit.measure_all()
    return circuit

def exact_probabilities(marked_bits, k):
    return Statevector.from_instruction(
        grover_circuit(n, marked_bits, k)
    ).probabilities_dict()

## 5. 🧪 Challenge 2 — Find the best number of Grover iterations

Move $k$ from 0 to 7. Watch the target probability rise, peak, and then fall. Can you find the best $k$ before looking at the theoretical answer?

In [ ]:
theta = math.asin(1 / math.sqrt(N))
k_rule = math.floor((math.pi / 4) * math.sqrt(N))
k_slider = widgets.IntSlider(
    value=k_rule, min=0, max=7, step=1, description='Grover k:',
    continuous_update=False, style={'description_width': 'initial'}
)
show_circuit = widgets.Button(description='Show circuit for this k')
iteration_output, circuit_output = widgets.Output(), widgets.Output()

def iteration_interpretation(k, probabilities):
    best_k = int(np.argmax(probabilities))
    if k == 0: return 'No amplification yet.'
    if k == best_k: return 'Near the optimum — the state is closest to the target.'
    if k < best_k: return 'Target amplified, but the rotation can move closer.'
    return 'Past the first optimum — Grover keeps rotating and can overshoot the target.'

def update_iteration_dashboard(change=None):
    _, marked_bits = current_target()
    k = k_slider.value
    probs = exact_probabilities(marked_bits, k)
    labels = [format(i, f'0{n}b') for i in range(N)]
    values = np.array([probs.get(label, 0) for label in labels])
    all_theory = np.sin((2 * np.arange(8) + 1) * theta) ** 2
    iteration_output.clear_output(wait=True)
    with iteration_output:
        display(Markdown(
            f"### Target $|{marked_bits}\rangle$  \n"
            f"## Success probability: {probs.get(marked_bits, 0):.1%}"
        ))
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.bar(labels, values,
               color=['#ef8354' if x == marked_bits else '#4f6d9a' for x in labels])
        ax.set(ylim=(0, 1), xlabel='row ID (binary)', ylabel='exact probability')
        plt.show()
        display(Markdown(f"**k = {k}:** {iteration_interpretation(k, all_theory)}"))

def draw_current_circuit(_):
    _, marked_bits = current_target()
    circuit_output.clear_output(wait=True)
    with circuit_output:
        display(grover_circuit(n, marked_bits, k_slider.value).draw('mpl', style='iqp', fold=-1))

k_slider.observe(update_iteration_dashboard, names='value')
target_dropdown.observe(update_iteration_dashboard, names='value')
show_circuit.on_click(draw_current_circuit)
display(k_slider, show_circuit, iteration_output, circuit_output)
update_iteration_dashboard()

## 6. 📐 Reveal the theory

After $k$ iterations, $P_k=\sin^2((2k+1)\theta)$ where $\theta=\arcsin(1/\sqrt N)$. Reveal this only after exploring.

In [ ]:
reveal_theory = widgets.Button(description='Reveal theoretical curve', button_style='info')
theory_output = widgets.Output()

def show_theory(_):
    ks = np.arange(8)
    theory = np.sin((2 * ks + 1) * theta) ** 2
    theory_output.clear_output(wait=True)
    with theory_output:
        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.plot(ks, theory, 'o-', color='#563d7c')
        ax.axvline(k_rule, color='#ef8354', linestyle='--', label=f'rule of thumb: k={k_rule}')
        ax.set(xticks=ks, ylim=(0, 1.05), xlabel='Grover iterations k',
               ylabel='ideal success probability', title='Grover amplification oscillates')
        ax.legend()
        display(fig)
        plt.close(fig)
        display(Markdown(
            f'**For N={N}, the rule of thumb gives k={k_rule}.**  \n'
            'Did the peak you found experimentally match the theory?'
        ))

reveal_theory.on_click(show_theory)
display(reveal_theory, theory_output)

## 7. 🎲 Measure it like a quantum computer

The exact statevector is a **simulator-only microscope**. Quantum hardware returns sampled bit strings. Choose a shot count and compare prediction with observation; repeated runs vary because of sampling noise.

In [ ]:
shots_selector = widgets.ToggleButtons(options=[10, 100, 1000], value=100, description='Shots:')
run_measurement = widgets.Button(description='🎲 Run measurement', button_style='success')
measurement_output = widgets.Output()
simulator = AerSimulator()

def sample_current_experiment(_):
    marked_id, marked_bits = current_target()
    k, shots = k_slider.value, shots_selector.value
    prediction_prob = exact_probabilities(marked_bits, k).get(marked_bits, 0)
    measured = grover_circuit(n, marked_bits, k, measure=True)
    counts = simulator.run(transpile(measured, simulator), shots=shots).result().get_counts()
    target_hits = counts.get(marked_bits, 0)
    observed = target_hits / shots
    winner = max(counts, key=counts.get)
    winner_row = rows[int(winner, 2)]
    measurement_output.clear_output(wait=True)
    with measurement_output:
        display(Markdown(
            f"### Statevector prediction: `{prediction_prob:.2%}`  \n"
            f"### Observed frequency:     `{observed:.2%}`"
        ))
        display(plot_histogram(counts, title=f'{shots:,} simulated measurements'))
        display(Markdown(
            f"**Measured target:** {target_hits} / {shots} times  \n"
            f"**Empirical success rate:** {observed:.1%}  \n"
            f"**Most frequent result:** $|{winner}\rangle$  \n"
            f"**Decoded row:** {winner_row['city']}, sales={winner_row['sales']}"
        ))

run_measurement.on_click(sample_current_experiment)
display(widgets.HBox([shots_selector, run_measurement]), measurement_output)

## 8. ✅ Final check & takeaway

Why can three Grover iterations be worse than two for $N=8$?

In [ ]:
final_question = widgets.RadioButtons(options=[
    'Each oracle call introduces random error',
    'Grover amplification oscillates and can rotate past the target',
    'Measuring more than once reduces the amplitude',
], value=None, layout=widgets.Layout(width='100%', height='120px'))
check_final = widgets.Button(description='Check answer', button_style='primary')
final_output = widgets.HTML()

def check_final_answer(_):
    if final_question.value is None:
        final_output.value = 'Choose an answer first.'
        return
    correct = final_question.value == 'Grover amplification oscillates and can rotate past the target'
    verdict = ('<h3>✅ Correct</h3>' if correct else
               '<h3>Not quite — think of each Grover step as another rotation.</h3>')
    final_output.value = verdict + '''<ul>
      <li>The <b>oracle</b> recognizes an answer and marks it with a phase flip.</li>
      <li>The <b>diffuser</b> turns that phase difference into larger target amplitude.</li>
      <li>Repeating both about O(√N) times gives a quadratic <b>query-complexity</b> improvement.</li>
      <li>Amplification oscillates: too many iterations rotate past the target.</li>
      <li>This is not a free SQL speedup: oracle construction, data loading, error correction, and readout carry end-to-end costs.</li>
    </ul>'''

check_final.on_click(check_final_answer)
display(widgets.VBox([final_question, check_final, final_output]))